# Notebook 4: Teacher -> MobileNetV3 Student Distillation

Requires Notebook 2's `stage1_crop_classifier.pt` and Notebook 3's 6 `stage2_<crop>_disease.pt` checkpoints on Drive. Distills each teacher (EfficientNetV2-S) into a MobileNetV3-Large student — 7 distillations total (1 crop classifier + 6 disease heads), one notebook, per-target resumable.

Uses the same group-aware train/val split as Notebook 3 (images grouped by inferred source photo, namespaced by label) — including for Stage-1, whose original split in Notebook 2 predates that fix and carries the same duplicate-leakage risk.

Distillation loss = `ALPHA * hard-label cross-entropy + (1 - ALPHA) * T^2 * KL-divergence(student soft logits, teacher soft logits)`, standard Hinton-style knowledge distillation.

## Task 1: Setup — load data + all teacher checkpoints

In [ ]:
!pip install -q timm pillow pandas scikit-learn gdown

In [ ]:
import os
import gdown

# In Google Drive: right-click the crop_disease folder -> Share -> General
# access -> "Anyone with the link" (Viewer) -> Copy link. Paste it below.
DRIVE_FOLDER_URL = "PASTE_YOUR_SHARED_FOLDER_LINK_HERE"
LOCAL_DIR = "/kaggle/working/crop_disease"

assert "PASTE_YOUR" not in DRIVE_FOLDER_URL, "Paste your shared Drive folder link above first"
gdown.download_folder(DRIVE_FOLDER_URL, output=LOCAL_DIR, quiet=False, use_cookies=False)
print("Downloaded:", os.listdir(LOCAL_DIR))

In [ ]:
import zipfile
import pandas as pd

DATA_ZIP = os.path.join(LOCAL_DIR, "data.zip")
UNIFIED_ROOT = "/content/data"  # matches the absolute paths already baked into manifest.csv's filepath column
STAGE_DIR = LOCAL_DIR
STAGE1_CKPT = os.path.join(STAGE_DIR, "stage1_crop_classifier.pt")

assert os.path.exists(DATA_ZIP), f"{DATA_ZIP} not found in {LOCAL_DIR} — check the shared folder actually contains data.zip"
assert os.path.exists(STAGE1_CKPT), f"{STAGE1_CKPT} not found — check the shared folder contains stage1_crop_classifier.pt"

os.makedirs(UNIFIED_ROOT, exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as zf:
    zf.extractall(UNIFIED_ROOT)

manifest = pd.read_csv(os.path.join(UNIFIED_ROOT, "manifest.csv"))
CROPS = sorted(manifest["crop"].unique())

for crop in CROPS:
    p = os.path.join(STAGE_DIR, f"stage2_{crop}_disease.pt")
    assert os.path.exists(p), f"{p} not found — check the shared folder contains all 6 stage2 checkpoints"

print(manifest.shape)
print("Crops:", CROPS)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Task 2: Shared utilities — dataset, transforms, group-aware split key, teacher loader

In [ ]:
import re
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ImageLabelDataset(Dataset):
    def __init__(self, df, label_col, label_to_idx, transform):
        self.df = df.reset_index(drop=True)
        self.label_col = label_col
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        label = self.label_to_idx[row[self.label_col]]
        return self.transform(img), label

# Same heuristic as Notebook 3: strip known augmentation-prefix patterns to
# recover a shared "source photo" key, namespaced by label so unrelated
# images with coincidentally identical filenames (seen in the citrus source)
# never merge into one group.
AUG_PREFIX_RE = re.compile(
    r"^(?:[a-z0-9]+_primary_|multi_crop_supplement_)"
    r"(?:resized_|rotated_|zoomed_|cropped_|flipped_horiz_|flipped_vert_|flipped_)*",
    re.IGNORECASE,
)

def build_group_key(filepath, label):
    stem = os.path.splitext(os.path.basename(filepath))[0]
    stripped = AUG_PREFIX_RE.sub("", stem, count=1)
    return f"{label}::{stripped.lower()}"

In [ ]:
def load_teacher(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    if "crop_to_idx" in ckpt:
        classes = sorted(ckpt["crop_to_idx"], key=ckpt["crop_to_idx"].get)
    else:
        classes = ckpt["diseases"]
    model = timm.create_model("tf_efficientnetv2_s", pretrained=False, num_classes=len(classes))
    model.load_state_dict(ckpt["model_state"])
    model = model.to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, classes

## Task 3: Distillation training function

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import GroupShuffleSplit

STUDENT_EPOCHS = 8
ALPHA = 0.5       # weight on hard-label loss vs soft-label (distillation) loss
TEMPERATURE = 4.0  # softmax temperature for distillation

def distill(target_name, df, label_col, teacher_ckpt_path, student_ckpt_path):
    teacher, classes = load_teacher(teacher_ckpt_path)
    recomputed = sorted(df[label_col].unique())
    assert recomputed == classes, (
        f"[{target_name}] class mismatch: manifest gives {recomputed}, teacher checkpoint has {classes} "
        "— data must have changed since the teacher was trained"
    )
    label_to_idx = {c: i for i, c in enumerate(classes)}

    df = df.copy()
    df["group_key"] = [build_group_key(fp, lbl) for fp, lbl in zip(df["filepath"], df[label_col])]
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
    train_idx, val_idx = next(gss.split(df, groups=df["group_key"]))
    train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]

    train_loader = DataLoader(ImageLabelDataset(train_df, label_col, label_to_idx, train_tfms), batch_size=32, shuffle=True, num_workers=2, drop_last=len(train_df) > 32)
    val_loader = DataLoader(ImageLabelDataset(val_df, label_col, label_to_idx, eval_tfms), batch_size=32, shuffle=False, num_workers=2)

    class_counts = train_df[label_col].map(label_to_idx).value_counts().sort_index()
    weights = (1.0 / class_counts.reindex(range(len(classes)), fill_value=1)).values
    class_weights = torch.tensor(weights * len(classes) / weights.sum(), dtype=torch.float32).to(DEVICE)

    student = timm.create_model("mobilenetv3_large_100", pretrained=True, num_classes=len(classes)).to(DEVICE)

    optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, STUDENT_EPOCHS * len(train_loader)))
    hard_criterion = nn.CrossEntropyLoss(weight=class_weights)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    start_epoch = 0
    best_val_acc = 0.0
    if os.path.exists(student_ckpt_path):
        ckpt = torch.load(student_ckpt_path, map_location=DEVICE)
        if ckpt.get("classes") == classes and ckpt.get("split_version") == "distill_v1":
            student.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            start_epoch = ckpt["epoch"] + 1
            best_val_acc = ckpt.get("val_acc", 0.0)
            print(f"  [{target_name}] resumed from epoch {start_epoch}, best_val_acc={best_val_acc:.4f}")
        else:
            print(f"  [{target_name}] existing student checkpoint is stale, retraining from scratch")

    if start_epoch >= STUDENT_EPOCHS:
        print(f"  [{target_name}] already fully trained ({STUDENT_EPOCHS} epochs), skipping")
        return

    for epoch in range(start_epoch, STUDENT_EPOCHS):
        student.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                with torch.no_grad():
                    teacher_logits = teacher(imgs)
                student_logits = student(imgs)
                hard_loss = hard_criterion(student_logits, labels)
                soft_loss = F.kl_div(
                    F.log_softmax(student_logits / TEMPERATURE, dim=1),
                    F.softmax(teacher_logits / TEMPERATURE, dim=1),
                    reduction="batchmean",
                ) * (TEMPERATURE ** 2)
                loss = ALPHA * hard_loss + (1 - ALPHA) * soft_loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running_loss += loss.item() * imgs.size(0)
            correct += (student_logits.argmax(1) == labels).sum().item()
            total += imgs.size(0)
        train_loss, train_acc = running_loss / total, correct / total

        student.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = student(imgs)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += imgs.size(0)
        val_acc = val_correct / val_total

        print(f"  [{target_name}] epoch {epoch+1}/{STUDENT_EPOCHS} train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        best_val_acc = max(best_val_acc, val_acc)
        torch.save({
            "model_state": student.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "val_acc": val_acc,
            "classes": classes,
            "label_col": label_col,
            "split_version": "distill_v1",
        }, student_ckpt_path)

    print(f"  [{target_name}] done. best_val_acc={best_val_acc:.4f}. Checkpoint: {student_ckpt_path}")

## Task 4: Run all 7 distillations (Stage-1 + 6x Stage-2)

In [ ]:
print("=== stage1_crop ===")
distill(
    "stage1_crop",
    manifest,
    "crop",
    STAGE1_CKPT,
    os.path.join(STAGE_DIR, "student_stage1_crop_classifier.pt"),
)

for crop_name in CROPS:
    print(f"\n=== stage2_{crop_name} ===")
    crop_df = manifest[manifest["crop"] == crop_name]
    distill(
        f"stage2_{crop_name}",
        crop_df,
        "disease",
        os.path.join(STAGE_DIR, f"stage2_{crop_name}_disease.pt"),
        os.path.join(STAGE_DIR, f"student_stage2_{crop_name}_disease.pt"),
    )

If this cell stops partway (GPU quota, disconnect), just rerun it — each `distill()` call skips targets whose student checkpoint is already at `STUDENT_EPOCHS` and resumes any target that's partway through.

## Task 5: Teacher vs student comparison — accuracy and model size

In [ ]:
def param_count(model):
    return sum(p.numel() for p in model.parameters())

targets = [("stage1_crop", STAGE1_CKPT, os.path.join(STAGE_DIR, "student_stage1_crop_classifier.pt"))]
targets += [
    (f"stage2_{c}", os.path.join(STAGE_DIR, f"stage2_{c}_disease.pt"), os.path.join(STAGE_DIR, f"student_stage2_{c}_disease.pt"))
    for c in CROPS
]

rows = []
for target_name, teacher_path, student_path in targets:
    teacher_ckpt = torch.load(teacher_path, map_location=DEVICE)
    student_ckpt = torch.load(student_path, map_location=DEVICE)
    classes = student_ckpt["classes"]

    teacher_model, _ = load_teacher(teacher_path)
    student_model = timm.create_model("mobilenetv3_large_100", pretrained=False, num_classes=len(classes))
    student_model.load_state_dict(student_ckpt["model_state"])

    rows.append({
        "target": target_name,
        "num_classes": len(classes),
        "teacher_val_acc": teacher_ckpt["val_acc"],
        "student_val_acc": student_ckpt["val_acc"],
        "teacher_params_M": round(param_count(teacher_model) / 1e6, 2),
        "student_params_M": round(param_count(student_model) / 1e6, 2),
    })

summary = pd.DataFrame(rows)
summary["acc_delta"] = summary["student_val_acc"] - summary["teacher_val_acc"]
print(summary.to_string(index=False))

`teacher_params_M` / `student_params_M` are raw parameter counts, not on-device file size — the real size reduction comes from INT8 quantization in Notebook 5, not from this parameter-count difference alone. Watch `acc_delta`: a large negative number means the student lost real accuracy going small, worth more distillation epochs or a bigger student (e.g. `mobilenetv3_large_100` -> `efficientnet_lite2`) before shipping.

## Task 6: Verify all 7 student checkpoints load standalone

In [ ]:
for target_name, _, student_path in targets:
    ckpt = torch.load(student_path, map_location=DEVICE)
    classes = ckpt["classes"]
    model = timm.create_model("mobilenetv3_large_100", pretrained=False, num_classes=len(classes))
    model.load_state_dict(ckpt["model_state"])
    model = model.to(DEVICE).eval()
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    with torch.no_grad():
        out = model(dummy)
    assert out.shape == (2, len(classes)), f"{target_name}: expected (2, {len(classes)}), got {out.shape}"
    print(f"{target_name}: OK, {len(classes)} classes")